# Reproducing Vittani et al. (2025) from dandiset 001414

**Paper.** Vittani M, Herlo R, Wang X, et al. (2025). Cerebral blood flow is modulated by astrocytic cAMP
elevation independently of IP3R2-mediated Ca2+ signaling in mice. *PNAS* 122, e2422069122.
[doi:10.1073/pnas.2422069122](https://doi.org/10.1073/pnas.2422069122). The claims file
(`claims/vittani-2025.yaml`) was written from the version of record (Europe PMC, PMC12260423) with
the SI Appendix.

**Dandiset.** [DANDI:001414](https://dandiarchive.org/dandiset/001414/0.250603.1324), version
0.250603.1324. The paper cites no version of the dandiset. This is its only published version; the
draft has the same 500 assets and size (350.8 GB) and differs only in later metadata edits, so the
most recent published version is pinned.

**What the dandiset holds.** Raw one-photon movies (uint16, 512 x 512, 5 Hz) for every LED experiment
of Figures 1 and 3 to 6 and Figure S1 (sensory stimulation), arteriole movies with their kymograph
rectangles, two-photon stacks (Figure S5), and single images. It holds no traces, no stimulus times,
no ROIs for the astrocyte signals and no results. Every number below is computed from the movies.

**This notebook** tests the astrocyte cAMP (Pink Flamindo) and Ca2+ (RCaMP3) claims. Each movie
(about 640 MB) is streamed chunk by chunk and reduced to its mean intensity per frame as it loads.
The arteriole claims are not tested (see the table and the record): the "rotated" arteriole movies
the rectangles were drawn on hold only 0 and 1, so the rectangles cannot be placed on the raw movies
without recovering each rotation, and streaming the 115 GB of arteriole movies did not fit the run.

| Claim | Tested here |
|---|---|
| fig1b-camp-dose-dependent, fig1g-j-camp-kinetics-anesthetized, fig1f-camp-reproducible | yes |
| fig3b-camp-awake, fig3cd-camp-awake-stable, awake-vs-anesthetized-camp-kinetics | yes |
| fig4a-e-camp-ip3r2-no-difference, fig4-camp-ip3r2-values | yes |
| fig5b-d-ca-anesthetized, figs4f-g-ca-rise-decay-anesthetized, fig5e-g-ca-awake, fig5h-j-ca-ip3r2-reduced | yes |
| fig6c-e-camp-mild, fig6f-h-ca-mild, figs1p-whisker-camp-length | yes |
| fig1k-astrocyte-expression, figs1k-twophoton-camp, figs5i-twophoton-ca-camp-correlation | blocked, metadata insufficient (data shown) |
| figs1n-startle-camp-awake-higher, figs4z-startle-ca-ip3r2 | blocked, metadata insufficient (no stimulus times) |
| all arteriole claims and the claims that depend on them | blocked, too large for the run |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
from statsmodels.stats.multitest import multipletests

import vittani as v

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
warnings.filterwarnings("ignore")
started = time.time()

assets = v.list_assets()
print(f"dandiset {v.DANDISET_ID} version {v.DANDISET_VERSION}: {len(assets)} assets, "
      f"{assets['size'].sum() / 1e9:.1f} GB")
print(assets.groupby("experiment").agg(files=("path", "size"), subjects=("subject", "nunique"),
                                       GB=("size", lambda s: round(s.sum() / 1e9, 1))).to_string())

In [ ]:
ASTRO = ["camp_kx", "camp_awake", "camp_ip3r2", "camp_mild", "ca_kx", "ca_awake", "ca_ip3r2", "ca_mild",
         "whisker_camp"]
sel = assets[assets.experiment.isin(ASTRO)]
t0 = time.time()
traces = v.load_traces(sel, workers=2)
print(f"streamed {len(sel)} files ({sel['size'].sum() / 1e9:.1f} GB) in {(time.time() - t0) / 60:.1f} min")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
meta = assets.set_index("path")
info = pd.DataFrame([{
    "path": p, "experiment": meta.loc[p, "experiment"], "subject": meta.loc[p, "subject"],
    "group": meta.loc[p, "group"], "genotype": meta.loc[p, "genotype"], "recording": meta.loc[p, "recording"],
    "identifier": r["identifier"], "indicator": r["indicator"], "rate_Hz": r["imaging_rate"],
    "series": ";".join(f"{k.replace('OnePhotonSeries', '')}{list(s)}" for k, s in r["series"].items()),
    "no_movie": bool(r.get("empty")), "frames": 0 if r.get("empty") else len(r["roi"])} for p, r in traces.items()])
print("Files, subjects and recordings per experiment and group (no_movie = the file's movie has fewer than 100 frames):")
print(info.groupby(["experiment", "group", "genotype"], dropna=False).agg(
    files=("path", "size"), subjects=("subject", "nunique"), no_movie=("no_movie", "sum"),
    frames=("frames", lambda f: sorted(set(f)))).to_string())
print("\nIndicator by experiment:")
print(info.groupby("experiment").indicator.unique().to_string())
print("\nImaging rate (Hz) in every file:", sorted(info.rate_Hz.unique()))
print("\nFiles with more than one movie, or none:")
print(info[info.series.str.count(";") > 0][["path", "series"]].to_string(index=False))
print(info[info.no_movie][["path", "series"]].to_string(index=False))

The identifiers inside the files (`fig_1_pf_kx_...`, `fig_5_rcamp3_awake_...`) agree with the experiment
read from each path. The counts match the legends: Figure 1, 4 bPAC mice (PF1 has no fourth recording,
as the legend's "3 mice for recording 4" says) and 5 controls; Figure 3, 5 and 5; Figure 4, 6 IP3R2-/-
and 6 IP3R2+/+; Figure 5, 5 and 5 (anesthetized and awake) and 4 IP3R2-/- with 3 IP3R2+/+. The mild
stimulation cAMP data (Figure 6) hold recordings 1 and 6 of 5 bPAC mice, not four recordings; the
movie of PF5's recording 6 is stored in PF5's recording 1 file with only 2 frames, and its own file is
empty, so 4 mice have both recordings. The mild Ca2+ data hold 4 bPAC mice (Ca11, 13, 15, 17) and 5
controls. The movie of PF584's recording 2 (Figure 4) is stored with time as the last axis
(512 x 512 x 1600, in a series named only `OnePhotonSeries`), unlike every other movie; it is read
with its axes moved. The subject age of the file `sub-Ca581-IP3R2-WT` recording 4 (P14.3W) differs from its
other recordings (P20.3W).

In [ ]:
# LED frames: the blue light shows as a step in the red channel. Onset and number of lit frames per file.
rows = []
for p, r in traces.items():
    if r.get("empty") or meta.loc[p, "experiment"] == "whisker_camp":
        continue
    led = v.led_frames(r["roi"])
    rows.append({"experiment": meta.loc[p, "experiment"], "led_s": meta.loc[p, "led_s"],
                 "onset": None if led is None else led[0], "lit": None if led is None else led[1] - led[0] + 1})
L = pd.DataFrame(rows)
print("LED onset frame and number of lit frames, by experiment and stimulation length (counts):")
print(L.groupby(["experiment", "led_s"]).agg(files=("onset", "size"), found=("onset", "count"),
      onset=("onset", lambda x: sorted(x.dropna().astype(int).unique().tolist())),
      lit=("lit", lambda x: sorted(x.dropna().astype(int).unique().tolist()))).to_string())

# Recordings deposited twice: identical mean-intensity traces
keys = [p for p, r in traces.items() if not r.get("empty")]
sig = {}
dups = []
for p in keys:
    k = tuple(np.round(traces[p]["roi"][:200], 3))
    if k in sig:
        dups.append((sig[k], p))
    sig.setdefault(k, p)
print("\nPairs of files with identical traces:", len(dups))
for a_, b_ in dups:
    pk = []
    for q in (a_, b_):
        t_, y_ = v.dff(traces[q]["roi"], traces[q]["blood_offset"], v.led_frames(traces[q]["roi"]), bleach=True)
        pk.append(y_[(t_ >= 0) & (t_ < 60)].max())
    print(f"   {a_} ({meta.loc[a_, 'group'] or meta.loc[a_, 'genotype']})\n   = {b_} ({meta.loc[b_, 'group'] or meta.loc[b_, 'genotype']}); "
          f"peak dF/F0 in the first 60 s {pk[0]:.1f}% and {pk[1]:.1f}%")

ranges = pd.DataFrame([{"experiment": meta.loc[p, "experiment"], "min": traces[p]["roi"].min(),
                        "max": traces[p]["roi"].max(), "blood_offset": traces[p]["blood_offset"]} for p in keys])
print("\nROI mean intensity (raw units) and Blood Offset, range by experiment:")
print(ranges.groupby("experiment").agg(["min", "max"]).round(0).to_string())

The files carry no stimulus times. The 65 s baseline of the Methods puts the LED at frame 325; the
step the LED light leaves in the red channel is found at frames 325 or 326 wherever it is visible, and
its length matches the stated durations (about 5 frames for 1 s, 1 for 0.2 s, 15 for 3 s). This is
consistent with the description; the response then rises after the step in bPAC mice and not in
controls (below), which shows the step marks the stimulation. Where no step is seen (most 25 ms, 1 mW
recordings and some controls), frame 325 is used.

Eight pairs of files hold identical traces, all in the awake cAMP experiments (Figures 3 and 4) and
one anesthetized control (PF5, whose 1 s and 0.2 s files are the same movie). Some pairs join two
mice, and two join a control or IP3R2+/+ mouse with a bPAC or IP3R2-/- mouse. Each movie is counted
once below: the second file of each pair (in path order) is left out, so the mouse the movie truly
belongs to cannot be told and some mice lose recordings. The results with every file kept are shown
as a variant where the claim tests recordings within mice.

In [ ]:
DUPLICATES = {b_ for a_, b_ in dups}
FILT = {"camp_kx": 20, "camp_mild": 20, "ca_kx": 20, "ca_mild": 20, "whisker_camp": 20,
        "camp_awake": 50, "camp_ip3r2": 50, "ca_awake": 50, "ca_ip3r2": 50}


def params_for(experiment):
    tr = {p: r for p, r in traces.items() if meta.loc[p, "experiment"] == experiment and p not in DUPLICATES}
    led = experiment != "whisker_camp"
    return v.astro_params(tr, assets, FILT[experiment], bleach=experiment.startswith(("camp", "whisker")), led=led)


P = {e: params_for(e) for e in ASTRO}
MEASURES = ["peak", "rise", "decay", "duration"]


def within(value, expected, tol):
    return abs(value - expected) <= tol


def compare(table, expected):
    """table: rows = readings, columns = quantities; expected: {quantity: (value, tolerance)}.
    Returns the table with the count of values within tolerance per reading, best first."""
    out = table.copy()
    out["n_within"] = sum(within(out[q], *expected[q]).astype(int) for q in expected)
    return out.sort_values("n_within", ascending=False)


print({e: (len(p_), p_.subject.nunique()) for e, p_ in P.items()})

## fig1b-camp-dose-dependent

> In anesthetized mice, blue LED stimulation of bPAC raises astrocytic Pink Flamindo fluorescence
> (cAMP) in a dose-dependent way across 0.2, 1 and 3 s stimulation.

Peak after the LED per mouse; 1 s is the mean of recordings 1 and 4 per mouse (Table S1). Baseline
readings (a) and (b) are both shown, with the peak on the filtered and the unfiltered trace.

In [ ]:
kx = P["camp_kx"].copy()
kx["stim"] = kx.recording.map({1: "1 s", 4: "1 s", 2: "0.2 s", 3: "3 s"})
per_mouse = kx.groupby(["reading", "group", "stim", "subject"])[MEASURES].mean().reset_index()
peaks = per_mouse.assign(F0=per_mouse.reading.str[:3], peak_on=per_mouse.reading.str[4:9])
tab = peaks[peaks.reading.str.endswith("decayb-t0a")].groupby(["F0", "peak_on", "group", "stim"]).peak.agg(
    ["mean", "sem", "count"]).unstack("stim")
print("Peak dF/F0 (%) by group and stimulation length; mean, SEM and mice:")
print(tab.round(2).to_string())
res = []
for (f0, pk), g in peaks[peaks.reading.str.endswith("decayb-t0a")].groupby(["F0", "peak_on"]):
    m = g[g.group == "bPAC"].groupby("stim").peak.mean()
    c = g[g.group == "ctrl"].groupby("stim").peak.mean()
    t_, p_ = stats.ttest_ind(g[(g.group == "bPAC") & (g.stim == "1 s")].peak, g[(g.group == "ctrl") & (g.stim == "1 s")].peak)
    res.append({"F0": f0, "peak_on": pk, "all above 0": bool((m > 0).all()),
                "0.2<1<3 s": bool(m["0.2 s"] < m["1 s"] < m["3 s"]), "bPAC 1 s": m["1 s"], "ctrl 1 s": c["1 s"],
                "bPAC>ctrl at 1 s": bool(m["1 s"] > c["1 s"]), "t (1 s)": t_, "p (1 s)": p_})
print("\nCriterion per reading:")
print(pd.DataFrame(res).round(4).to_string(index=False))
print("\nPer mouse, bPAC, peak (%) under F0a-peakf:")
print(peaks[(peaks.reading == "F0a-peakf-decayb-t0a") & (peaks.group == "bPAC")].pivot(index="subject", columns="stim", values="peak").round(1).to_string())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for i, (rec, lab) in enumerate([(2, "0.2 s"), (1, "1 s (rec 1)"), (3, "3 s")]):
    for grp, col in (("bPAC", "C3"), ("ctrl", "0.5")):
        paths = [p for p in traces if meta.loc[p, "experiment"] == "camp_kx" and meta.loc[p, "group"] == grp
                 and meta.loc[p, "recording"] == rec]
        t, y = v.mean_trace(traces, paths, bleach=True, led_s={1: 1.0, 2: 0.2, 3: 3.0}[rec])
        ax[i].plot(t, y, color=col, label=f"{grp} (n={len(paths)})")
        print(f"{lab} {grp}: mean trace range {y.min():.1f} to {y.max():.1f} %")
    ax[i].set_title(lab); ax[i].set_xlabel("s from the first frame after the LED"); ax[i].legend()
ax[0].set_ylabel("Pink Flamindo dF/F0 (%)")
plt.tight_layout(); plt.show()

**Result: reproduced.** Under all four readings the bPAC mean peak is above zero at every length and rises from 0.2 s to 1 s to 3 s (79.6, 104.7 and 113.1% with baseline (a) and filtered peaks), and at 1 s the bPAC mice (104.7%) are far above the 5 controls (2.5%; t = 11.44). PF6 has the largest responses (113 to 151%); PF1 is the one mouse whose 3 s peak (87.9%) is below its 1 s peak (93.8%).

## fig1g-j-camp-kinetics-anesthetized

> Longer LED stimulation gives higher cAMP peaks, faster rise times, slower decay and longer responses,
> with the values of Table S1 (0.2 s, 1 s, 3 s at 5 mW in anesthetized WT mice).

Every combination of the open readings (baseline F0 a/b, peak on filtered/unfiltered trace, 80% decay
a/b, time origin a/b) is shown, with the count of the 12 values within tolerance and the ordering of
the four parameters. The paired t tests with Benjamini-Hochberg correction over the pairs of
recordings follow, for the two families the criterion names.

In [ ]:
EXP_1G = {("peak", "0.2 s"): (67.2, 12.8), ("peak", "1 s"): (94.4, 4.4), ("peak", "3 s"): (106.6, 17.6),
          ("rise", "0.2 s"): (7.9, 0.9), ("rise", "1 s"): (6.3, 0.2), ("rise", "3 s"): (0.45, 0.2),
          ("decay", "0.2 s"): (30.7, 2.5), ("decay", "1 s"): (62.2, 4.4), ("decay", "3 s"): (95.4, 7.1),
          ("duration", "0.2 s"): (38.6, 3.1), ("duration", "1 s"): (68.5, 4.2), ("duration", "3 s"): (95.8, 7.1)}
b = per_mouse[per_mouse.group == "bPAC"]
means = b.groupby(["reading", "stim"])[MEASURES].mean().unstack("stim")
rows = []
for rd, m in means.iterrows():
    order = (m[("peak", "0.2 s")] < m[("peak", "1 s")] < m[("peak", "3 s")]
             and m[("rise", "0.2 s")] > m[("rise", "1 s")] > m[("rise", "3 s")]
             and m[("decay", "0.2 s")] < m[("decay", "1 s")] < m[("decay", "3 s")]
             and m[("duration", "0.2 s")] < m[("duration", "1 s")] < m[("duration", "3 s")])
    n = sum(within(m[k], *e) for k, e in EXP_1G.items())
    rows.append({"reading": rd, "ordering": order, "n_within": n, **{f"{k[0]} {k[1]}": m[k] for k in EXP_1G}})
R1 = pd.DataFrame(rows).sort_values(["ordering", "n_within"], ascending=False)
print("Expected:", {f"{k[0]} {k[1]}": e for k, e in EXP_1G.items()})
print(R1.round(2).to_string(index=False))
best_1g = R1.iloc[0]
print("\nBest reading:", best_1g.reading, "ordering", best_1g.ordering, "within", best_1g.n_within, "of 12")
print("SEM over mice under the best reading:")
print(b[b.reading == best_1g.reading].groupby("stim")[MEASURES].sem().round(2).to_string())

In [ ]:
def paired_bh(df, reading, recs=(1, 2, 3, 4)):
    d = df[(df.reading == reading) & (df.group == "bPAC")]
    out = []
    for m in MEASURES:
        w = d.pivot(index="subject", columns="recording", values=m)
        for i, a_ in enumerate(recs):
            for b_ in recs[i + 1:]:
                pair = w[[a_, b_]].dropna()
                t_, p_ = stats.ttest_rel(pair[a_], pair[b_])
                out.append({"measure": m, "pair": f"{a_} vs {b_}", "n": len(pair), "diff": (pair[b_] - pair[a_]).mean(),
                            "t": t_, "p": p_})
    out = pd.DataFrame(out)
    out["q_within_measure"] = out.groupby("measure").p.transform(lambda p: multipletests(p, method="fdr_bh")[1])
    out["q_all"] = multipletests(out.p, method="fdr_bh")[1]
    return out


PT = paired_bh(P["camp_kx"], best_1g.reading)
print("Paired t tests between recordings (1 = 1 s, 2 = 0.2 s, 3 = 3 s, 4 = 1 s), reading", best_1g.reading)
print(PT.round(4).to_string(index=False))

**Result: reproduced.** Under the reading F0a-peakf-decayb-t0a (baseline over the whole 65 s, peak on the median-filtered trace, decay to 20% of the peak, time from the first frame after the LED), the ordering holds for all four parameters and 9 of the 12 values are within tolerance; the 0.2 s and 1 s rise times (6.35 and 4.35 s against 7.9 and 6.3) and the 1 s peak (104.7 against 94.4%) are outside. The same reading with baseline (b) also gives 9 of 12. With "80% decay" read as the fall to 80% of the peak (decay a), at most 3 values match, so the Table S1 decay and duration values fit reading (b). By the paired t tests with Benjamini-Hochberg correction, every pair of different lengths differs in decay and duration (q at most 0.0067), and 0.2 s differs from 1 s and 3 s in peak (q 0.0052).

## fig1f-camp-reproducible

> The 1 s stimulation gave similar cAMP kinetic profiles at the first and the last recording, showing
> the stimulation is reproducible.

The paper's test: paired t tests of recording 1 against 4 with Benjamini-Hochberg correction, on the
3 mice with both, for the four parameters, under each reading.

In [ ]:
rows = []
for rd in sorted(P["camp_kx"].reading.unique()):
    pt = paired_bh(P["camp_kx"], rd)
    s = pt[pt.pair == "1 vs 4"]
    rows.append({"reading": rd, **{f"{m} diff": s[s.measure == m]["diff"].iloc[0] for m in MEASURES},
                 **{f"{m} p": s[s.measure == m].p.iloc[0] for m in MEASURES},
                 "min q (a, within measure)": s.q_within_measure.min(), "min q (b, all)": s.q_all.min(),
                 "n mice": int(s.n.iloc[0])})
R1F = pd.DataFrame(rows)
print(R1F.round(4).to_string(index=False))
print("\nReadings where any parameter differs at q < 0.05: family (a)", int((R1F["min q (a, within measure)"] < 0.05).sum()),
      "; family (b)", int((R1F["min q (b, all)"] < 0.05).sum()), "of", len(R1F))

**Result: reproduced, resting on the reading adopted in fig1g-j-camp-kinetics-anesthetized.** Under F0a-peakf-decayb-t0a recording 1 and recording 4 (3 mice) do not differ in any parameter (smallest q 0.20 in family (a), 0.21 in family (b)); the peak is 13.4% lower in recording 4 and the decay and duration 1.9 and 1.5 s shorter. With decay read as the fall to 80% of the peak and filtered peaks, the decay and duration differ (q 0.03 to 0.05 in family (a) for 4 of the 16 readings), so the outcome depends on the reading.

## fig3b-camp-awake

> In awake mice, 1 s LED stimulation raises Pink Flamindo fluorescence significantly more in bPAC
> mice than in control mice, peaking at about 92.7% after about 3 s and decaying over about 78 s, a
> response of about 81 s.

Per mouse, the mean over its recordings (1 to 4); then the mean over mice. Awake median filter of 50
frames. The test is an unpaired t test of per-mouse peaks, bPAC against control.

In [ ]:
EXP_3B = {"peak": (92.7, 1.6), "rise": (3.3, 0.3), "decay": (78.0, 1.8), "duration": (81.4, 1.6)}
aw = P["camp_awake"].groupby(["reading", "group", "subject"])[MEASURES].mean().reset_index()
rows = []
for rd, g in aw.groupby("reading"):
    m = g[g.group == "bPAC"][MEASURES].mean()
    t_, p_ = stats.ttest_ind(g[g.group == "bPAC"].peak, g[g.group == "ctrl"].peak)
    rows.append({"reading": rd, **m.to_dict(), "ctrl peak": g[g.group == "ctrl"].peak.mean(), "t": t_, "p": p_,
                 "n_within": sum(within(m[k], *e) for k, e in EXP_3B.items()),
                 "n bPAC": int((g.group == "bPAC").sum()), "n ctrl": int((g.group == "ctrl").sum())})
R3 = pd.DataFrame(rows).sort_values("n_within", ascending=False)
print("Expected:", EXP_3B)
print(R3.round(4).to_string(index=False))
best_3b = R3.iloc[0]
print("\nPer mouse under", best_3b.reading)
print(aw[aw.reading == best_3b.reading].round(2).to_string(index=False))

**Result: partially reproduced.** The bPAC peak is far above the control peak under every reading (90.3 against 7.3%, t = 14.34, p < 0.001, 5 and 5 mice), but at most one of the four values is within tolerance under any reading. Under F0a-peakf-decayb-t0a the duration (81.4 s) matches, the decay (75.7 s) and the peak (90.3%) fall just outside their tolerances, and the rise time (5.7 s against 3.3) is outside. Six of the 39 files duplicate others (see the data check) and are left out.

## fig3cd-camp-awake-stable

> In awake WT mice the cAMP peak, rise time, decay time and response duration do not differ across the
> four 1 s recordings.

The paper's test is a mixed-effects model (REML) across recordings. PF1 has three recordings, so the
repeated-measures ANOVA with Geisser-Greenhouse correction on the 4 mice with all four is shown, and a
linear mixed model (mouse as random intercept, recording as fixed factor, REML) on all 5 mice, tested
by the Wald chi-square of the recording terms.

In [ ]:
import statsmodels.formula.api as smf


def rm_tests(df, reading, groups=("bPAC",)):
    warnings.simplefilter("ignore")
    d = df[(df.reading == reading) & (df.group.isin(groups))]
    out = []
    for m in MEASURES:
        full = d.groupby("subject").recording.nunique()
        complete = d[d.subject.isin(full[full == full.max()].index)]
        try:
            a_ = pg.rm_anova(data=complete, dv=m, within="recording", subject="subject", correction=True)
        except ZeroDivisionError:         # fewer complete mice than the test needs
            a_ = pd.DataFrame({"F": [np.nan], "p_unc": [np.nan]})
        pcol = "p_GG_corr" if "p_GG_corr" in a_ else "p_unc"
        try:
            if complete.subject.nunique() == d.subject.nunique():
                raise ValueError("no missing recordings; the RM ANOVA is the test")
            fit = smf.mixedlm(f"{m} ~ C(recording)", d, groups=d["subject"]).fit(reml=True)
            names = [n for n in fit.params.index if n.startswith("C(recording)")]
            wald = fit.wald_test(" , ".join(f"{n} = 0" for n in names), scalar=True)
            p_mixed = float(wald.pvalue)
        except Exception as err:          # noqa: BLE001
            p_mixed = np.nan
        means = d.groupby("recording")[m].mean()
        out.append({"measure": m, "F": float(a_["F"].iloc[0]), "p_GG": float(a_[pcol].iloc[0]),
                    "eps": float(a_["eps"].iloc[0]) if "eps" in a_ else np.nan, "n complete": complete.subject.nunique(),
                    "p_mixed (all mice)": p_mixed, "n all": d.subject.nunique(),
                    "rec means": " / ".join(f"{x:.1f}" for x in means)})
    return pd.DataFrame(out)


rows = []
for rd in sorted(P["camp_awake"].reading.unique()):
    r_ = rm_tests(P["camp_awake"], rd)
    rows.append({"reading": rd, **{f"{m} p_GG": r_[r_.measure == m].p_GG.iloc[0] for m in MEASURES},
                 **{f"{m} p_mixed": r_[r_.measure == m]["p_mixed (all mice)"].iloc[0] for m in MEASURES}})
R3C = pd.DataFrame(rows)
print(R3C.round(4).to_string(index=False))
print("\nReadings with any p < 0.05: RM ANOVA", int((R3C.filter(like="p_GG") < 0.05).any(axis=1).sum()),
      "; mixed model", int((R3C.filter(like="p_mixed") < 0.05).any(axis=1).sum()), "of", len(R3C))
print("\nDetail under", best_3b.reading)
print(rm_tests(P["camp_awake"], best_3b.reading).round(4).to_string(index=False))
P_keep = v.astro_params({p: r for p, r in traces.items() if meta.loc[p, "experiment"] == "camp_awake"}, assets, 50, bleach=True)
print("\nVariant with every file kept (duplicates counted as deposited), under", best_3b.reading)
print(rm_tests(P_keep, best_3b.reading).round(4).to_string(index=False))
print("Adopted reading for the four parameters (fig1g-j): F0a-peakf-decayb-t0a")
print(rm_tests(P["camp_awake"], "F0a-peakf-decayb-t0a").round(4).to_string(index=False))
print(rm_tests(P_keep, "F0a-peakf-decayb-t0a").round(4).to_string(index=False))

**Result: not reproduced.** The paper's test, a mixed-effects model across recordings, finds the recordings to differ in 10 of the 16 readings; under F0a-peakf-decayb-t0a the peak (recording means 91.9, 93.2, 86.0 and 84.2%), rise time and decay time differ (p 0.0031, 0.0025 and 0.018), and the duration does not (p 0.17). Three of the five bPAC mice lose recordings to the duplicate files and PF1 has three, so only one mouse has all four and the repeated-measures ANOVA cannot be run on the de-duplicated data; the variant with every file kept is printed above. The difference may come from the duplicated deposits, not from the paper.

## awake-vs-anesthetized-camp-kinetics

> For 1 s stimulation, cAMP rises faster and decays more slowly in awake than in anesthetized WT mice,
> with similar peaks.

The same reading for both groups: anesthetized 1 s (mean of recordings 1 and 4 per mouse), awake mean
of recordings 1 to 4 per mouse.

In [ ]:
an = per_mouse[(per_mouse.group == "bPAC") & (per_mouse.stim == "1 s")].groupby("reading")[MEASURES].agg(["mean", "sem"])
awm = aw[aw.group == "bPAC"].groupby("reading")[MEASURES].agg(["mean", "sem"])
rows = []
for rd in an.index:
    a_, w_ = an.loc[rd], awm.loc[rd]
    rows.append({"reading": rd, "rise anesth": a_[("rise", "mean")], "rise awake": w_[("rise", "mean")],
                 "decay anesth": a_[("decay", "mean")], "decay awake": w_[("decay", "mean")],
                 "peak anesth": a_[("peak", "mean")], "peak awake": w_[("peak", "mean")],
                 "peak diff / anesth peak": abs(w_[("peak", "mean")] - a_[("peak", "mean")]) / a_[("peak", "mean")],
                 "SEMs anesth, awake": f"{a_[('peak', 'sem')]:.1f}, {w_[('peak', 'sem')]:.1f}",
                 "faster rise awake": w_[("rise", "mean")] < a_[("rise", "mean")],
                 "slower decay awake": w_[("decay", "mean")] > a_[("decay", "mean")]})
RAV = pd.DataFrame(rows)
RAV["holds"] = RAV["faster rise awake"] & RAV["slower decay awake"] & (RAV["peak diff / anesth peak"] < 0.10)
print(RAV.round(3).to_string(index=False))
print("\nReadings where all three parts hold:", int(RAV.holds.sum()), "of", len(RAV))

**Result: not reproduced, resting on the reading adopted in fig1g-j-camp-kinetics-anesthetized.** Under F0a-peakf-decayb-t0a the awake decay is longer (75.7 against 62.1 s), but the rise is slower in awake mice (5.7 against 4.35 s) and the peaks differ by 13.8% of the anesthetized peak (90.3 against 104.7%). All three parts hold under the 4 readings that take the peak on the unfiltered trace with decay (b) (rise 4.5 against 5.7 s, decay 73.4 against 60.4 s, peaks 103.0 and 106.4%, 3.2% apart).

## fig4-camp-ip3r2-values

> Pink Flamindo peaked at 7.2 s at 62.2% and lasted 69.5 s in IP3R2-/- mice, and peaked at 8.4 s at
> 54.4% and lasted 85.1 s in IP3R2+/+ mice.

Means over mice of each mouse's mean over recordings 1 to 4.

In [ ]:
EXP_4 = {("KO", "peak"): (62.2, 1.3), ("KO", "rise"): (7.2, 1.1), ("KO", "decay"): (62.3, 12.0), ("KO", "duration"): (69.5, 11.1),
         ("WT", "peak"): (54.4, 1.5), ("WT", "rise"): (8.4, 0.7), ("WT", "decay"): (76.7, 11.1), ("WT", "duration"): (85.1, 11.3)}
ip = P["camp_ip3r2"].groupby(["reading", "genotype", "subject"])[MEASURES].mean().reset_index()
gm = ip.groupby(["reading", "genotype"])[MEASURES].mean().unstack("genotype")
rows = []
for rd, m in gm.iterrows():
    rows.append({"reading": rd, **{f"{g} {q}": m[(q, g)] for g, q in EXP_4},
                 "n_within": sum(within(m[(q, g)], *e) for (g, q), e in EXP_4.items())})
R4 = pd.DataFrame(rows).sort_values("n_within", ascending=False)
print("Expected:", {f"{g} {q}": e for (g, q), e in EXP_4.items()})
print(R4.round(2).to_string(index=False))
best_4 = R4.iloc[0]
print("\nMice per genotype:", ip[ip.reading == best_4.reading].genotype.value_counts().to_dict())
print("Per mouse under", best_4.reading)
print(ip[ip.reading == best_4.reading].round(2).to_string(index=False))

**Result: not reproduced.** At most 2 of the 8 values are within tolerance under any reading. Under F0a-peakf-decayb-t0a the IP3R2-/- and IP3R2+/+ peaks are 57.4 and 63.2% (expected 62.2 and 54.4, so the order of the genotypes is reversed), the rise times 7.7 and 8.0 s (both within tolerance), and the decay times 41.5 and 41.6 s and durations 49.2 and 49.7 s, well below the 62 to 85 s of Table S1. One pair of the 48 files is identical, joining an IP3R2-/- and an IP3R2+/+ mouse.

## fig4a-e-camp-ip3r2-no-difference

> In awake IP3R2-/- and IP3R2+/+ littermates, 1 s stimulation raises astrocytic cAMP with no difference
> between genotypes in peak, rise time, decay time or response duration.

The paper's test: two-way ANOVA, genotype between mice, recording within mice. `pingouin.mixed_anova`
uses mice with all four recordings; the genotype effect on the per-mouse means of all 12 mice (an
unpaired t test, the same as the between-mice term of a balanced design) is shown beside it.

In [ ]:
rows = []
for rd in sorted(P["camp_ip3r2"].reading.unique()):
    d = P["camp_ip3r2"][P["camp_ip3r2"].reading == rd]
    full = d.groupby("subject").recording.nunique()
    comp = d[d.subject.isin(full[full == 4].index)]
    r_ = {"reading": rd}
    for m in MEASURES:
        a_ = pg.mixed_anova(data=comp, dv=m, within="recording", between="genotype", subject="subject")
        g_ = a_[a_.Source == "genotype"].iloc[0]
        r_[f"{m} F"] = g_["F"]; r_[f"{m} p"] = g_["p_unc"]
        pm = d.groupby(["subject", "genotype"])[m].mean().reset_index()
        r_[f"{m} p (all 12, t)"] = stats.ttest_ind(pm[pm.genotype == "KO"][m], pm[pm.genotype == "WT"][m]).pvalue
    r_["df"] = f"1, {comp.subject.nunique() - 2}"
    rows.append(r_)
R4A = pd.DataFrame(rows)
print(R4A.round(4).to_string(index=False))
print("\nReadings with a genotype effect at p < 0.05 (two-way ANOVA):",
      int((R4A[[f"{m} p" for m in MEASURES]] < 0.05).any(axis=1).sum()), "of", len(R4A),
      "; by the t test on all 12 mice:", int((R4A[[f"{m} p (all 12, t)" for m in MEASURES]] < 0.05).any(axis=1).sum()))

**Result: reproduced, resting on the reading adopted in fig1g-j-camp-kinetics-anesthetized.** Under F0a-peakf-decayb-t0a the two-way ANOVA finds no genotype effect on any parameter (p 0.44 to 0.96, df 1, 9), and the t tests on all 12 mice agree (p 0.59 to 0.98). With decay read as the fall to 80% of the peak and the unfiltered peak (4 of the 16 readings), the duration differs between genotypes (p 0.022 to 0.027), so the outcome depends on the reading.

## fig5b-d-ca-anesthetized

> In anesthetized WT mice bPAC stimulation raises astrocytic RCaMP3 (Ca2+) signals, and peak amplitude
> and response duration change across the successive recordings, the peaks falling, likely from
> depletion of intracellular Ca2+ stores.

RCaMP3 as the cAMP pipeline without photobleaching correction; anesthetized median filter of 20.

In [ ]:
EXP_5B = {("peak", 2): (235.4, 18.4), ("peak", 3): (192.3, 12.5), ("peak", "1+4"): (199.3, 45.8),
          ("duration", 2): (26.7, 3.4), ("duration", 3): (51.8, 7.8), ("duration", "1+4"): (35.6, 8.2)}
EXP_S4F = {("rise", 2): (4.1, 0.3), ("rise", 3): (2.4, 0.2), ("rise", "1+4"): (3.0, 0.2),
           ("decay", 2): (22.6, 2.6), ("decay", 3): (49.4, 7.9), ("decay", "1+4"): (32.6, 8.1)}
cak = P["ca_kx"]


def ca_kx_table(expected):
    rows = []
    for rd, d in cak[cak.group == "bPAC"].groupby("reading"):
        w = d.pivot_table(index="subject", columns="recording", values=MEASURES)
        r_ = {"reading": rd}
        for (q, rec), e in expected.items():
            val = (w[q][[1, 4]].mean(axis=1) if rec == "1+4" else w[q][rec]).mean()
            r_[f"{q} {rec}"] = val
        r_["n_within"] = sum(within(r_[f"{q} {rec}"], *e) for (q, rec), e in expected.items())
        rows.append(r_)
    return pd.DataFrame(rows).sort_values("n_within", ascending=False)


R5 = ca_kx_table(EXP_5B)
print("Expected:", EXP_5B)
print(R5.round(2).to_string(index=False))
best_5 = R5.iloc[0].reading
pk = cak[cak.reading == best_5].pivot_table(index=["group", "subject"], columns="recording", values="peak")
print("\nPeak (%) per mouse and recording under", best_5)
print(pk.round(1).to_string())
gm5 = pk.groupby("group").mean()
print("bPAC above control in every recording:", bool((gm5.loc["bPAC"] > gm5.loc["ctrl"]).all()),
      "; recording 4 below recording 1 (bPAC mean):", bool(gm5.loc["bPAC", 4] < gm5.loc["bPAC", 1]))
rows = []
for rd in sorted(cak.reading.unique()):
    r_ = rm_tests(cak, rd)
    rows.append({"reading": rd, **{f"{m} F": r_[r_.measure == m].F.iloc[0] for m in MEASURES},
                 **{f"{m} p_GG": r_[r_.measure == m].p_GG.iloc[0] for m in MEASURES}})
R5T = pd.DataFrame(rows)
print("\nRepeated-measures ANOVA (Geisser-Greenhouse) across the four recordings, bPAC mice:")
print(R5T.round(4).to_string(index=False))

**Result: partially reproduced.** The bPAC RCaMP3 peak exceeds the control peak in every recording (controls are negative, -14 to 3%), and recording 4 is below recording 1. The repeated-measures ANOVA finds the peak to change across recordings under every reading (p 0.017 to 0.028); the duration changes under 8 of the 16 readings, including those with the unfiltered peak and decay (b) (p 0.0016 to 0.040), but not under F0a-peakf-decayb-t0a (p 0.085). At most 2 of the 6 values are within tolerance under any reading: the 0.2 s peak (225.1 against 235.4%) and the mean 1 s peak (181.3 against 199.3%) match, while the 3 s peak (153.5%) and the durations (16 to 26 s against 27 to 52 s) are lower.

## figs4f-g-ca-rise-decay-anesthetized

> In anesthetized mice RCaMP3 rise time and decay time differ with stimulation length; the text says
> shorter LED stimulation gives faster rise times, while Table S4 gives 4.1 s for 0.2 s, 3.0 s for 1 s
> and 2.4 s for 3 s.

In [ ]:
RS4 = ca_kx_table(EXP_S4F)
print("Expected:", EXP_S4F)
print(RS4.round(2).to_string(index=False))
best_s4 = RS4.iloc[0].reading
print("\nRepeated-measures ANOVA for rise and decay under", best_s4)
print(R5T[R5T.reading == best_s4][["reading", "rise F", "rise p_GG", "decay F", "decay p_GG"]].round(4).to_string(index=False))
print("Readings where both ANOVAs give p < 0.05:", int(((R5T["rise p_GG"] < 0.05) & (R5T["decay p_GG"] < 0.05)).sum()), "of", len(R5T))
r_ = RS4.iloc[0]
print(f"Rise time direction under {best_s4}: 0.2 s {r_['rise 2']:.2f}, 1 s {r_['rise 1+4']:.2f}, 3 s {r_['rise 3']:.2f} s")

**Result: not reproduced.** At most 1 of the 6 values is within tolerance under any reading; the 3 s decay time is at most 30.3 s under any reading, against 49.4 s in Table S4. Both ANOVAs give p < 0.05 under 4 of the 16 readings. Under the reading that matches one value, the rise time grows with stimulation length (0.2 s 2.68, 1 s 2.82, 3 s 3.12 s), the direction of the sentence and not of Table S4.

## fig5e-g-ca-awake

> In awake WT mice, 1 s stimulation raises RCaMP3 signals in bPAC mice, peaking at about 2 s and
> decaying over about 38 s, a response of about 40 s; peaks decline across recordings while response
> duration does not change.

In [ ]:
EXP_5E = {"peak": (184.8, 16.4), "rise": (1.78, 0.2), "decay": (37.9, 1.4), "duration": (39.71, 1.4)}
caw = P["ca_awake"]
cam = caw.groupby(["reading", "group", "subject"])[MEASURES].mean().reset_index()
rows = []
for rd, g in cam.groupby("reading"):
    m = g[g.group == "bPAC"][MEASURES].mean()
    t_ = rm_tests(caw, rd)
    recm = caw[(caw.reading == rd) & (caw.group == "bPAC")].groupby("recording").peak.mean()
    rows.append({"reading": rd, **m.to_dict(), "ctrl peak": g[g.group == "ctrl"].peak.mean(),
                 "n_within": sum(within(m[k], *e) for k, e in EXP_5E.items()),
                 **{f"{q} p_GG": t_[t_.measure == q].p_GG.iloc[0] for q in MEASURES},
                 "rec4 < rec1": bool(recm[4] < recm[1])})
R5E = pd.DataFrame(rows).sort_values("n_within", ascending=False)
R5E["tests hold"] = (R5E["peak p_GG"] < 0.05) & R5E["rec4 < rec1"] & (R5E[["duration p_GG", "rise p_GG", "decay p_GG"]] >= 0.05).all(axis=1)
print("Expected:", EXP_5E)
print(R5E.round(4).to_string(index=False))

**Result: not reproduced.** The bPAC response is there (peak 70.1% under F0a-peakf-decayb-t0b against 15.4% in controls; 210.5% on the unfiltered trace), but the peak does not change significantly across recordings under any reading (p 0.054 to 0.145), though recording 4 is below recording 1, and at most one of the four values is within tolerance (the rise time, 1.70 s). Decay and duration (24.9 and 26.6 s) are shorter than the 37.9 and 39.7 s of Table S4.

## fig5h-j-ca-ip3r2-reduced

> The bPAC-induced astrocytic Ca2+ elevation is much smaller, shorter and slightly delayed in IP3R2-/-
> than in IP3R2+/+ mice (peak 28.7 against 115.2%, rise time 2.4 against 1.8 s, response 7.8 against
> 22.2 s), and in each group the peaks fall over successive recordings.

In [ ]:
EXP_5H = {("KO", "peak"): (28.7, 4.6), ("WT", "peak"): (115.2, 12.2), ("KO", "rise"): (2.4, 0.3),
          ("WT", "rise"): (1.8, 0.2), ("KO", "duration"): (7.8, 0.2), ("WT", "duration"): (22.2, 4.0)}
cai = P["ca_ip3r2"]
rows = []
for rd, d in cai.groupby("reading"):
    pm = d.groupby(["genotype", "subject"])[MEASURES].mean().reset_index()
    gmean = pm.groupby("genotype")[MEASURES].mean()
    r_ = {"reading": rd, **{f"{g} {q}": gmean.loc[g, q] for g, q in EXP_5H}}
    r_["n_within"] = sum(within(gmean.loc[g, q], *e) for (g, q), e in EXP_5H.items())
    for m in MEASURES:
        a_ = pg.mixed_anova(data=d, dv=m, within="recording", between="genotype", subject="subject")
        r_[f"{m} genotype F"] = a_[a_.Source == "genotype"].F.iloc[0]
        r_[f"{m} genotype p"] = a_[a_.Source == "genotype"].p_unc.iloc[0]
    a_ = pg.mixed_anova(data=d, dv="peak", within="recording", between="genotype", subject="subject")
    r_["peak recording p"] = a_[a_.Source == "recording"].p_unc.iloc[0]
    recm = d.groupby(["genotype", "recording"]).peak.mean()
    r_["rec4<rec1 KO, WT"] = f"{recm['KO'][4] < recm['KO'][1]}, {recm['WT'][4] < recm['WT'][1]}"
    rows.append(r_)
R5H = pd.DataFrame(rows).sort_values("n_within", ascending=False)
print("Expected:", {f"{g} {q}": e for (g, q), e in EXP_5H.items()})
print(R5H.round(4).to_string(index=False))
print("\nMice per genotype:", cai[cai.reading == R5H.iloc[0].reading].groupby("genotype").subject.nunique().to_dict())

**Result: partially reproduced.** The genotype effects on decay and duration are significant with IP3R2-/- shorter (F0b-peakf-decayb-t0b, decay p 0.0003, duration p 0.0006; durations 7.8 and 22.3 s, both within tolerance), and the IP3R2-/- peak is below the IP3R2+/+ peak (11.0 against 41.9%). Only 2 of the 6 values match: the peaks are lower than Table S4 (28.7 and 115.2%). The genotype effect on peak is significant here (p 0.022), where the paper's is not. The rise time is shorter in IP3R2-/- mice (1.66 against 3.38 s, p 0.022), the reverse of "slightly delayed". The recording effect on peak is significant (p 0.0039), with recording 4 below recording 1 in both genotypes.

## fig6c-e-camp-mild

> Mild bPAC stimulation (25 ms, 1 mW) in anesthetized mice raises astrocytic cAMP by about 8%, peaking
> at about 6 s and lasting about 28 s, similar to sensory-evoked cAMP; the paired t tests of the
> legends find no difference between the recordings compared.

The dandiset holds recordings 1 and 6 for these mice, the pair Table S1 names ("#1 and 6"); the
claims file's readings (a) recordings 1 and 4 and (b) all four cannot be formed. Values are the mean
of recordings 1 and 6 per mouse, and the paired t tests compare recording 1 with 6.

In [ ]:
EXP_6C = {"peak": (7.8, 0.6), "rise": (5.73, 0.2), "decay": (22.4, 3.4), "duration": (28.2, 3.4)}
cm = P["camp_mild"]
print("Recordings per mouse:", cm.groupby("subject").recording.apply(lambda r: sorted(set(r))).to_dict())
rows = []
for rd, d in cm.groupby("reading"):
    w = d.pivot_table(index="subject", columns="recording", values=MEASURES)
    both = w.dropna()
    m = d[d.subject.isin(both.index)].groupby("subject")[MEASURES].mean().mean()
    r_ = {"reading": rd, **m.to_dict(), "n_within": sum(within(m[k], *e) for k, e in EXP_6C.items()), "n": len(both)}
    for q in MEASURES:
        t_, p_ = stats.ttest_rel(both[q][1], both[q][6])
        r_[f"{q} t"] = t_; r_[f"{q} p"] = p_
    m_all = d.groupby("subject")[MEASURES].mean().mean()
    r_["peak (all 5 mice)"] = m_all["peak"]
    rows.append(r_)
R6 = pd.DataFrame(rows).sort_values("n_within", ascending=False)
R6["tests all p >= 0.05"] = (R6[[f"{q} p" for q in MEASURES]] >= 0.05).all(axis=1)
print("Expected:", EXP_6C)
print(R6.round(4).to_string(index=False))

**Result: partially reproduced.** The dandiset holds recordings 1 and 6, the pair Table S1 names; 4 mice have both. Under F0a-peakf-decaya-t0a the peak (7.98%) and the duration (31.2 s) are within tolerance, 2 of the 4 values; no reading matches more. The rise time is 27 s under every reading, against 5.7 s: the filtered trace keeps rising after the first peak. None of the four paired t tests finds recording 1 and 6 to differ (p 0.31 to 0.50 under F0a-peakf-decayb-t0a). The 7.98% peak is close to the whisker peaks of figs1p-whisker-camp-length (4.7 to 6.7%).

## fig6f-h-ca-mild

> Mild bPAC stimulation raises astrocytic RCaMP3 signals in bPAC mice and not in controls, peaking at
> about 8.5 s at about 73% and lasting about 17 s, with peak and duration changing across recordings.

In [ ]:
EXP_6F = {"peak": (72.5, 13.5), "rise": (8.5, 0.5), "decay": (8.7, 0.2), "duration": (17.2, 0.6)}
cml = P["ca_mild"]
print("Mice per group:", cml.groupby("group").subject.nunique().to_dict())
rows = []
for rd, d in cml.groupby("reading"):
    pm = d.groupby(["group", "subject"])[MEASURES].mean().reset_index()
    m = pm[pm.group == "bPAC"][MEASURES].mean()
    t_ = rm_tests(cml, rd)
    rows.append({"reading": rd, **m.to_dict(), "ctrl peak": pm[pm.group == "ctrl"].peak.mean(),
                 "n_within": sum(within(m[k], *e) for k, e in EXP_6F.items()),
                 "peak F": t_[t_.measure == "peak"].F.iloc[0], "peak p_GG": t_[t_.measure == "peak"].p_GG.iloc[0],
                 "duration F": t_[t_.measure == "duration"].F.iloc[0], "duration p_GG": t_[t_.measure == "duration"].p_GG.iloc[0]})
R6F = pd.DataFrame(rows).sort_values("n_within", ascending=False)
print("Expected:", EXP_6F)
print(R6F.round(4).to_string(index=False))
d = cml[cml.reading == R6F.iloc[0].reading]
print("\nPeak (%) per mouse and recording under", R6F.iloc[0].reading)
print(d.pivot_table(index=["group", "subject"], columns="recording", values="peak").round(1).to_string())

**Result: partially reproduced.** The 4 bPAC mice respond (peak 99.7% under F0a-peakf-decayb-t0a) and the 5 controls do not (-7.6%), and the peak and the duration change across recordings (p 0.014 and 0.0004). None of the four values is within tolerance under any reading: the peak (99.7 to 111.5%) is higher, and the rise (6.8 to 7.7 s) and duration (10.2 to 15.0 s) shorter, than Table S4 gives.

## figs1p-whisker-camp-length

> During air-puff whisker stimulation of anesthetized mice the cAMP response lasts through the
> stimulation, and its peak grows with stimulation length, to about 5% at 2 min (3 to 6% over the
> lengths used).

No stimulus times are in the files; the puffs start after the 65 s baseline (claims header, SI), at
frame 325. Peak over the recording after the onset. Photobleaching correction as for cAMP, and
without it as a variant.

In [ ]:
wk = P["whisker_camp"].rename(columns={"recording": "minutes"})
nob = v.astro_params({p: r for p, r in traces.items() if meta.loc[p, "experiment"] == "whisker_camp"},
                     assets, 20, bleach=False, led=False).rename(columns={"recording": "minutes"})
rows = []
for name, src in (("photobleaching corrected", wk), ("no correction", nob)):
    for rd, d in src.groupby("reading"):
        if not rd.endswith("decayb-t0a"):
            continue
        g = d.groupby("minutes").peak
        f_, p_ = stats.f_oneway(*[x.values for _, x in g])
        m = g.mean()
        rows.append({"pipeline": name, "reading": rd, **{f"{k} min": m[k] for k in m.index},
                     "n": "/".join(str(x) for x in g.size()), "rising": bool(m.is_monotonic_increasing),
                     "F": f_, "p": p_, "2 min within 5 +/- 0.5": within(m[2.0], 5, 0.5)})
RW = pd.DataFrame(rows)
print(RW.round(4).to_string(index=False))
print("\nPer mouse (photobleaching corrected, F0a-peakf):")
print(wk[wk.reading == "F0a-peakf-decayb-t0a"].pivot_table(index="subject", columns="minutes", values="peak").round(2).to_string())
for mins in (0.5, 1.0, 2.0):
    paths = [p for p in traces if meta.loc[p, "experiment"] == "whisker_camp" and meta.loc[p, "recording"] == mins]
    t, y = v.mean_trace(traces, paths, bleach=True, led=False, led_s=0)
    first = np.argmax(y[(t >= 0)]) if mins else 0
    seg = (t >= 0) & (t <= mins * 60)
    ipk = np.flatnonzero(t >= 0)[np.argmax(y[t >= 0][: int(mins * 60 * v.FPS)])]
    held = y[ipk:np.flatnonzero(t <= mins * 60)[-1] + 1]
    print(f"{mins} min: mean trace max {y[seg].max():.2f}% at {t[ipk]:.1f} s; above zero from that peak to the end "
          f"of the puffs: {bool((held > 0).all())} (min {held.min():.2f}%)")

**Result: not reproduced.** The mean peak does not rise with stimulation length (5.74, 4.74 and 6.70% for 0.5, 1 and 2 min with the photobleaching correction; ANOVA p = 0.55), and the 2 min mean is above the 5 +/- 0.5% tolerance. The 0.5 min mean rests on 3 mice, one of them at 10.45%. Against the 3 to 6% range, widened to 2.7 to 6.6%, the 0.5 and 1 min means lie inside and the 2 min mean outside. The response does persist: the mean trace stays above zero from its first peak to the end of the puffs for each length (minimum 1.66 to 2.53%). Without the photobleaching correction the peaks rise with length (ANOVA p 0.0003 to 0.0017) but the 0.5 min peaks are negative, because the bleaching drift is larger than the response.

## Blocked claims: what the dandiset holds for them

The histology image (Figure 1K) and the two-photon images (Figure S1J and K) are stored as single
frames without channel names or ROIs; the two-photon stacks of Figure S5 have two unnamed channels
(ChanA, ChanB) and astrocyte ROIs. They are listed here, read as metadata only.

In [ ]:
import h5py

for exp in ("other", "twophoton_gcamp", "startle_camp_awake", "startle_camp_kx", "startle_ca"):
    rows = assets[assets.experiment == exp]
    print(f"{exp}: {len(rows)} files, {rows['size'].sum() / 1e9:.1f} GB")
for p in assets[assets.experiment.isin(["other", "twophoton_gcamp"])].path[:8]:
    url, f = v.open_file(meta.loc[p, "asset_id"])
    ser = {k: (f["acquisition"][k]["data"].shape, f["acquisition"][k].attrs.get("description")) for k in f["acquisition"]}
    rois = f["processing/ophys/ImageSegmentation/PlaneSegmentation/id"].shape[0] if "processing/ophys" in f else 0
    print(p.split("/")[1][:70], ser, "ROIs:", rois, "| stimulus/intervals groups:",
          [k for k in ("stimulus/presentation", "intervals") if k in f and len(f[k])])
print("\nArteriole files: values in the first frame of each movie, and the kymograph rectangles")
for exp in ("alb_kx", "alb_awake", "alb_ip3r2", "alb_mild"):
    p = assets[assets.experiment == exp].path.iloc[0]
    url, f = v.open_file(meta.loc[p, "asset_id"])
    for k in f["acquisition"]:
        x = f["acquisition"][k]["data"][0]
        print(f"  {p.split('/')[1][:55]} {k.replace('OnePhotonSeries', '')}: {len(np.unique(x))} distinct values, "
              f"range {x.min()} to {x.max()}")
    m = f["processing/ophys/ImageSegmentation/PlaneSegmentation/image_mask"]
    print(f"    {m.shape[0]} rectangles drawn on {m.shape[1]} x {m.shape[2]} frames; no rotation angle is stored")
print("Arteriole movies:", int(assets.experiment.str.startswith("alb").sum()), "files,",
      round(assets[assets.experiment.str.startswith("alb")]["size"].sum() / 1e9, 1), "GB")
p = assets[assets.experiment == "startle_camp_awake"].path.iloc[0]
url, f = v.open_file(meta.loc[p, "asset_id"])
print("\nA startle file:", p, {k: f["acquisition"][k]["data"].shape for k in f["acquisition"]},
      "| stimulus/presentation:", list(f["stimulus/presentation"]), "| intervals:", list(f["intervals"]))

## Summary

| Claim | Outcome |
|---|---|
| fig1k-astrocyte-expression | blocked, metadata insufficient |
| fig1b-camp-dose-dependent | reproduced |
| fig1g-j-camp-kinetics-anesthetized | reproduced |
| fig1f-camp-reproducible | reproduced (assumption) |
| figs1n-startle-camp-awake-higher | blocked, metadata insufficient |
| figs1p-whisker-camp-length | not reproduced |
| figs1k-twophoton-camp | blocked, metadata insufficient |
| fig3b-camp-awake | partially reproduced |
| fig3cd-camp-awake-stable | not reproduced |
| awake-vs-anesthetized-camp-kinetics | not reproduced (assumption) |
| fig4a-e-camp-ip3r2-no-difference | reproduced (assumption) |
| fig4-camp-ip3r2-values | not reproduced |
| fig5b-d-ca-anesthetized | partially reproduced |
| figs4f-g-ca-rise-decay-anesthetized | not reproduced |
| fig5e-g-ca-awake | not reproduced |
| fig5h-j-ca-ip3r2-reduced | partially reproduced |
| figs4z-startle-ca-ip3r2 | blocked, metadata insufficient |
| figs5i-twophoton-ca-camp-correlation | blocked, metadata insufficient |
| fig6c-e-camp-mild | partially reproduced |
| fig6f-h-ca-mild | partially reproduced |
| 20 claims on the arterioles or resting on them (Figures 2, 3E to K, 4F to K, 6B and I to L, S1Q, S3H, S5A to D) | blocked, too large for the run |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

How the analysis differs from the paper's. The ROI over the thinned skull and the dark-vessel ROI of
the Blood Offset are not in the files and were chosen here (a central disc of radius 160 pixels, the
darkest 0.5% of its pixels in the first frame). No motion correction was applied. The across-recording
"Corrected dF/F0" was not applied, as the passage does not say how it is fitted. The LED frames were
found from the step the light leaves in the recording. Every open reading of the claims file was run;
where one had to be adopted, it is F0a-peakf-decayb-t0a, the reading under which the anesthetized cAMP
values best match Table S1.

What was found about the dataset. The files hold raw movies and nothing else for the astrocyte
signals: no traces, ROIs, stimulus times or results. Eight pairs of files hold identical movies,
some across mice and across groups. The mild-stimulation cAMP data hold recordings 1 and 6, and one
of them (PF5, recording 6) holds only 2 frames. One movie is stored with time as its last axis. The
"rotated" arteriole movies, on which the kymograph rectangles were drawn, hold only the values 0 and
1. The startle recordings and whisker recordings carry no stimulus times, and the two-photon stacks
do not say which channel is GCaMP6f and which Pink Flamindo.

In [ ]:
print(f"total run time {(time.time() - started) / 60:.1f} min")